## 1.Загрузка данных и первичный анализ

In [1]:
import urllib.request
import pandas as pd

def load_tweeteval_data(text_url, label_url):
    # Скачивание текстов
    with urllib.request.urlopen(text_url) as f:
        texts = f.read().decode('utf-8').strip().split('\n')
    # Скачивание меток
    with urllib.request.urlopen(label_url) as f:
        labels = [int(line.strip()) for line in f.read().decode('utf-8').strip().split('\n') if line.strip()]
    
    return pd.DataFrame({'text': texts, 'label': labels})

# Ссылки на сырые файлы в репозитории CardiffNLP
base_url = "https://raw.githubusercontent.com/cardiffnlp/tweeteval/main/datasets/sentiment/"
train_df = load_tweeteval_data(base_url + "train_text.txt", base_url + "train_labels.txt")
test_df = load_tweeteval_data(base_url + "test_text.txt", base_url + "test_labels.txt")

# Ограничим выборку для экономии времени обучения (опционально, например, до 2000 объектов)
train_df = train_df.sample(2000, random_state=42).reset_index(drop=True)
test_df = test_df.sample(500, random_state=42).reset_index(drop=True)

print(f"Размер обучающей выборки: {train_df.shape}")
print(f"Размер тестовой выборки: {test_df.shape}")
print("\nРаспределение классов (0 - Neg, 1 - Neu, 2 - Pos):")
print(train_df['label'].value_counts(normalize=True))
display(train_df.head())

Размер обучающей выборки: (2000, 2)
Размер тестовой выборки: (500, 2)

Распределение классов (0 - Neg, 1 - Neu, 2 - Pos):
label
1    0.4405
2    0.4055
0    0.1540
Name: proportion, dtype: float64


,text,label
0,I forgot all about Ice Cube being in the movie...,0
1,playoffs are finally set. Chardon plays warren...,1
2,Are we just going to ignore the fact that Ice ...,1
3,If you live in the South Orlando area\u002c be...,1
4,First record of Colin Baker at the BBC: BBC2 s...,1


## 2.Очистка и предобработка текста
Удалим лишние символы, ссылки (http), юзернеймы (@user) и знаки пунктуации

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # Приведение к нижнему регистру
    text = text.lower()
    # Удаление шаблонов @user и ссылок http
    text = re.sub(r"@user", "", text)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    # Удаление знаков пунктуации и спецсимволов
    text = re.sub(r"[^\w\s]", "", text)
    # Удаление стоп-слов
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

# Применяем очистку
train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)

print("Текст до очистки:", train_df['text'].iloc[0])
print("Текст после очистки:", train_df['clean_text'].iloc[0])

Текст до очистки: I forgot all about Ice Cube being in the movie First Sunday. I think I seen this shit in the theaters. 
Текст после очистки: forgot ice cube movie first sunday think seen shit theaters


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asizi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 3. Векторизация с помощью TF-IDF

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ограничим словарь до 3000 популярных слов, добавим биграммы (ngram_range=(1,2))
tfidf_vectorizer = TfidfVectorizer(max_features=3000, min_df=2, max_df=0.9, ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['clean_text']).toarray()
X_test_tfidf = tfidf_vectorizer.transform(test_df['clean_text']).toarray()

print(f"Форма TF-IDF матрицы для обучения: {X_train_tfidf.shape}")
print(f"Форма TF-IDF матрицы для тестирования: {X_test_tfidf.shape}")

Форма TF-IDF матрицы для обучения: (2000, 3000)
Форма TF-IDF матрицы для тестирования: (500, 3000)


## 4. Генерация и сохранение эмбеддингов

In [4]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer

# Автоопределение доступного ускорителя (CUDA для Nvidia, MPS для Apple, иначе CPU)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Используемое устройство для эмбеддингов: {device}")

# Передаем параметр device в модель
embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

# Генерируем эмбеддинги (передаем очищенный текст)
X_train_embed = embed_model.encode(train_df['clean_text'].tolist(), show_progress_bar=True)
X_test_embed = embed_model.encode(test_df['clean_text'].tolist(), show_progress_bar=True)

# Сохраняем эмбеддинги в файлы
np.save("X_train_embeddings.npy", X_train_embed)
np.save("X_test_embeddings.npy", X_test_embed)

print("Эмбеддинги успешно сгенерированы и сохранены в файлы!")
print(f"Форма матрицы эмбеддингов: {X_train_embed.shape}")

d:\Прога\ML\ML_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Используемое устройство для эмбеддингов: cpu


Batches: 100%|██████████| 16/16 [00:01<00:00, 12.91it/s]

Эмбеддинги успешно сгенерированы и сохранены в файлы!
Форма матрицы эмбеддингов: (2000, 384)


## 5. Обучение классификаторов и подбор гиперпараметров

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
import mlflow

y_train = train_df['label']
y_test = test_df['label']

# Функция для подбора параметров и логирования
def train_and_evaluate(X_tr, y_tr, X_te, y_te, feature_type):
    # Настройки сеток
    models_config = {
        'Logistic Regression': (
            LogisticRegression(max_iter=1000, class_weight='balanced'),
            {'C': [0.1, 1.0, 10.0]}
        ),
        'SVM': (
            SVC(class_weight='balanced'),
            {'C': [0.1, 1.0, 10.0], 'kernel': ['linear', 'rbf']}
        )
    }
    
    for name, (model, params) in models_config.items():
        with mlflow.start_run(run_name=f"{name}_{feature_type}"):
            grid = GridSearchCV(model, params, cv=3, scoring='f1_macro', n_jobs=-1)
            grid.fit(X_tr, y_tr)
            
            # Предсказание
            preds = grid.predict(X_te)
            acc = accuracy_score(y_te, preds)
            
            # Логирование
            mlflow.log_param("feature_type", feature_type)
            mlflow.log_params(grid.best_params_)
            mlflow.log_metric("test_accuracy", acc)
            
            print(f"--- {name} ({feature_type}) ---")
            print(f"Лучшие параметры: {grid.best_params_}")
            print(f"Accuracy на тесте: {acc:.4f}")
            print(classification_report(y_te, preds))

# Обучаем на признаках TF-IDF
train_and_evaluate(X_train_tfidf, y_train, X_test_tfidf, y_test, "TF-IDF")

# Обучаем на признаках Эмбеддингов
train_and_evaluate(X_train_embed, y_train, X_test_embed, y_test, "Embeddings")

--- Logistic Regression (TF-IDF) ---
Лучшие параметры: {'C': 1.0}
Accuracy на тесте: 0.5260
              precision    recall  f1-score   support

           0       0.51      0.37      0.43       158
           1       0.56      0.60      0.58       227
           2       0.47      0.59      0.53       115

    accuracy                           0.53       500
   macro avg       0.52      0.52      0.51       500
weighted avg       0.53      0.53      0.52       500

--- SVM (TF-IDF) ---
Лучшие параметры: {'C': 1.0, 'kernel': 'linear'}
Accuracy на тесте: 0.5060
              precision    recall  f1-score   support

           0       0.46      0.36      0.41       158
           1       0.54      0.59      0.56       227
           2       0.48      0.55      0.51       115

    accuracy                           0.51       500
   macro avg       0.49      0.50      0.49       500
weighted avg       0.50      0.51      0.50       500

--- Logistic Regression (Embeddings) ---
Лучшие па

## Ответы на заданные вопросы

1. **Какой датасет вы выбрали?**
   * Выбран поднабор `Sentiment` бенчмарка `TweetEval` (состоит из англоязычных твитов с тремя классами тональности: 0 — негативный, 1 — нейтральный, 2 — позитивный).
2. **Какой способ подготовки текста вы выбрали?**
   * Была проведена базовая предобработка: приведение к нижнему регистру, удаление специфических твиттер-тегов (юзернеймы `@user`, веб-ссылки `http/https`), удаление знаков пунктуации, а также фильтрация шумовых стоп-слов английского языка с помощью `nltk.corpus.stopwords`.
3. **Какой способ векторизации текста вы выбрали? Какие у него плюсы и минусы?**
   * Я сравнил два способа векторизации: **TF-IDF** и **Эмбеддинги (Sentence Transformers)**.
     * **TF-IDF:**
       * *Плюсы:* Считается мгновенно, не требует видеокарты, интерпретируем (мы четко знаем, какое слово дало какой вес).
       * *Минусы:* Не улавливает семантическую схожесть (слова «хороший» и «прекрасный» для него абсолютно не связаны), приводит к очень разреженным матрицам высокой размерности.
     * **Эмбеддинги:**
       * *Плюсы:* Передают смысл, синонимию, контекст предложения и грамматические структуры; дают плотные векторы фиксированной (сравнительно небольшой) размерности.
       * *Минусы:* Требуют значительных вычислительных мощностей (особенно при запуске тяжелых LLM/Transformer-моделей).
4. **Что такое эмбеддинг?**
   * **Эмбеддинг** — это представление текста (слова, предложения или целого документа) в виде плотного числового вектора фиксированной размерности (например, 384 или 768 чисел), полученного с помощью нейросетевых языковых моделей. Эмбеддинги строятся так, чтобы семантически похожие тексты находились близко друг к другу в многомерном векторном пространстве (по косинусному расстоянию).
5. **Как работает TF-IDF?**
   * Мера **TF-IDF** (Term Frequency – Inverse Document Frequency) оценивает важность слова для конкретного документа в коллекции. Она состоит из двух частей:
     * **TF (частота слова):** Отношение числа вхождений слова к общему количеству слов в документе (чем чаще встречается в документе, тем выше вес).
     * **IDF (обратная частота документов):** Логарифм отношения общего числа документов к числу документов, содержащих это слово. Она занижает вес слов, которые одинаково часто встречаются во всех текстах (например, предлоги, союзы или общеупотребительные термины).
6. **Какой классификатор вы выбрали? Почему? Какие гиперпараметры подбирали?**
   * Были выбраны классификаторы **Logistic Regression** и **SVM** (с линейным и RBF ядрами), так как они отлично работают на текстовых признаках высокой размерности. В качестве гиперпараметров подбирался коэффициент регуляризации `C`, определяющий баланс между сложностью модели и точностью на обучающей выборке, а также типы ядер для SVM.
7. **Как можно попробовать улучшить качество классификации? Где узкое место?**
   * *Узкое место:* В данной задаче узким местом чаще всего является **подготовка данных и выбор языковой модели**. Твиты полны сарказма, сленга и сокращений, которые базовая очистка и простые модели эмбеддингов улавливают с трудом.
   * *Как улучшить:* 
     1. Использовать специализированные предобученные на твитах нейросети (например, `cardiffnlp/twitter-roberta-base-sentiment`).
     2. Использовать лемматизацию (приведение слов к начальной форме) вместо простого удаления стоп-слов.
     3. Настроить глубокое дообучение (Fine-Tuning) трансформера, а не просто использовать его как экстрактор признаков.